# Sales analysis (sample)

A synthetic notebook written in the style of typical Databricks analytics notebooks, used to exercise `mborobak-scan`. It is not client code.

In [ ]:
%pip install some-package

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType
import pandas as pd
import os

In [ ]:
orders = spark.table("sales.orders")
customers = spark.read.table("sales.customers")
returns = spark.read.format("delta").load("/mnt/raw/returns")
display(orders.limit(10))

In [ ]:
clean = (
    orders
    .filter(F.col("status").isin("PAID", "SHIPPED"))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("amount", F.col("amount").cast(DecimalType(12, 2)))
    .withColumn("country", F.upper(F.trim(F.col("country"))))
    .withColumn("discount", F.coalesce(F.col("discount"), F.lit(0)))
    .dropDuplicates(["order_id"])
    .na.fill({"channel": "unknown"})
)

In [ ]:
monthly = (
    clean
    .withColumn("month", F.date_trunc("month", F.col("order_date")))
    .groupBy("country", "month")
    .agg(F.sum("amount").alias("revenue"),
         F.countDistinct("customer_id").alias("buyers"),
         F.round(F.avg("amount"), 2).alias("avg_ticket"))
)
monthly.cache()

In [ ]:
w = Window.partitionBy("country").orderBy(F.col("revenue").desc())
running = Window.partitionBy("country").orderBy("month").rowsBetween(Window.unboundedPreceding, Window.currentRow)
ranked = (
    monthly
    .withColumn("rank", F.row_number().over(w))
    .withColumn("cumulative", F.sum("revenue").over(running))
    .withColumn("prev", F.lag("revenue", 1).over(Window.partitionBy("country").orderBy("month")))
    .filter("rank <= 3")
)

In [ ]:
vip = (
    clean.join(customers, "customer_id", "left")
    .join(returns, on="order_id", how="left_anti")
    .where(F.col("segment") == "VIP")
    .select("customer_id", "name", F.concat_ws(" ", "first_name", "last_name").alias("full_name"),
            F.when(F.col("amount") > 1000, "high").otherwise("normal").alias("tier"))
    .selectExpr("*", "amount * 0.9 AS net")
)
vip = vip.withColumn("email_domain", F.regexp_extract("email", "@(.*)$", 1))

In [ ]:
%sql
SELECT country, SUM(revenue) FROM monthly GROUP BY country

In [ ]:
pdf = ranked.orderBy("country", "rank").toPandas()
pdf.head()
summary = pdf.groupby("country").sum()
print(", ".join(pdf.columns))
path = os.path.join("/tmp", "x")

In [ ]:
ranked.write.mode("overwrite").saveAsTable("sales.top_months")
dbutils.notebook.exit("done")